<a href="https://colab.research.google.com/github/2303A51876/NLP_project_Thulasi_Shylasri/blob/main/Cleaned%20dataset%20in%203%20languges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Import Libraries and Load Dataset***

In [1]:
# 1. IMPORT LIBRARIES AND LOAD DATASET

import pandas as pd
import numpy as np
import re
import openpyxl
from google.colab import files

print("Please select and upload 'Telangana_NLP_Stress_Sentiment_Dataset.xlsx':")
uploaded = files.upload()

# Load the uploaded file into DataFrame
file_path = list(uploaded.keys())[0]
df = pd.read_excel(file_path)

print(f"\n[INFO] Dataset Loaded Successfully!")
print(f"Original Dataset Shape: {df.shape}")
df.head(3)

Please select and upload 'Telangana_NLP_Stress_Sentiment_Dataset.xlsx':


Saving Telangana_NLP_Stress_Sentiment_Dataset.xlsx to Telangana_NLP_Stress_Sentiment_Dataset (1).xlsx

[INFO] Dataset Loaded Successfully!
Original Dataset Shape: (1500, 22)


,Candidate_ID,Candidate_Name,Age,Gender,State,District,Institution,Candidate_Type,Academic_Year,Raw_Text_Input,...,Sentiment_Score,Detected_Emotions,Stress_Level,Stress_Score,Primary_Issue,Issue_Category,PHQ8_Depression_Score,GAD7_Anxiety_Score,Crisis_Flag,Recommended_Support_Action
0,TS-EMH-000001,Teja Mudiraj,17,Male,Telangana,Hanamkonda,KITS Warangal (KITSW),Hostel Resident Student,1st Year,Previous semester lo 2 backlogs ochayi... Intl...,...,-0.90,"Sadness, Fear, Guilt",High,0.82,Backlog Stress & Fear,Academic,20,17,Yes,Academic Remedial Plan & Counseling Session
1,TS-EMH-000191,Harini Reddy,17,Female,Telangana,Hanamkonda,Kakatiya University (KU),Undergraduate Student,1st Year,Hanamkonda hostel lo undi complete single feel...,...,-0.58,"Sadness, Loneliness",Moderate,0.55,Hostel Isolation & Loneliness,Social Isolation,15,13,No,Peer Support Group Connection & Activity Schedule
2,TS-EMH-000387,Srikanth Yadav,17,Male,Telangana,Warangal,SR University (SRU),Undergraduate Student,1st Year,Companies visiting SRU/KITSW campus but my CGP...,...,-0.50,"Fear, Anxiety",High,0.90,Placement Rejection Fear,Career,16,16,No,Career Counseling & Resume Building Support


***Handle Missing Values and Duplicate Entries***

In [2]:
# 2. MISSING VALUES & DUPLICATES CHECK

# Drop duplicate records based on Candidate_ID
df.drop_duplicates(subset=['Candidate_ID'], inplace=True)

# Drop any potential missing/null values in key columns
critical_cols = ['Candidate_ID', 'Raw_Text_Input', 'Age', 'Gender', 'Institution']
df.dropna(subset=critical_cols, inplace=True)

print(f"[INFO] Dataset Shape after deduplication & missing value removal: {df.shape}")

[INFO] Dataset Shape after deduplication & missing value removal: (1500, 22)


***Text Normalization & Cleaning for NLP***

In [3]:
# 3. TEXT CLEANING FOR NLP PIPELINE

def clean_nlp_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML tags (<br>, <div>, etc.)
    text = re.sub(r'<[^>]+>', ' ', text)

    # 2. Remove web URLs (http, https, www)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. Remove hashtags and mentions symbols (# and @)
    text = re.sub(r'[#@]', '', text)

    # 4. Remove special characters keeping standard alphanumeric & basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)

    # 5. Normalize multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Create cleaned and lowercased text features
df['Cleaned_Text_Input'] = df['Raw_Text_Input'].apply(clean_nlp_text)
df['Cleaned_Text_Input_Lower'] = df['Cleaned_Text_Input'].str.lower()

print("[INFO] Sample Text Transformation:")
print("Original:", df['Raw_Text_Input'].iloc[0])
print("Cleaned :", df['Cleaned_Text_Input'].iloc[0])

[INFO] Sample Text Transformation:
Original: Previous semester lo 2 backlogs ochayi... Intlo parents ki cheppaleni bayam ga undi severe stress #backlog #telangana 
Cleaned : Previous semester lo 2 backlogs ochayi... Intlo parents ki cheppaleni bayam ga undi severe stress backlog telangana


***Standardize Categorical Data and Data Types***

In [4]:
# 4. STANDARDIZE CATEGORICAL & NUMERIC DATA

# Strip leading/trailing whitespaces from string columns
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Ensure integer types for clinical metrics and age
numeric_int_cols = ['Age', 'PHQ8_Depression_Score', 'GAD7_Anxiety_Score']
for col in numeric_int_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

# Ensure float types for scores
numeric_float_cols = ['Sentiment_Score', 'Stress_Score']
for col in numeric_float_cols:
    if col in df.columns:
        df[col] = df[col].astype(float)

# Bound clinical scores within standard diagnostic limits
if 'PHQ8_Depression_Score' in df.columns:
    df['PHQ8_Depression_Score'] = df['PHQ8_Depression_Score'].clip(lower=0, upper=24)

if 'GAD7_Anxiety_Score' in df.columns:
    df['GAD7_Anxiety_Score'] = df['GAD7_Anxiety_Score'].clip(lower=0, upper=21)

print("[INFO] Data types successfully standardized.")

[INFO] Data types successfully standardized.


***Validate Cleaned Data Integrity & Summary***

In [5]:
# 5. DATASET INTEGRITY VALIDATION

print("=== CLEANED DATASET SUMMARY ===")
print(f"Total Records : {len(df)}")
print(f"Total Columns : {len(df.columns)}")
print("\nUnique States:", df['State'].unique())
print("\nDistrict Distribution:")
print(df['District'].value_counts())
print("\nMissing Values Count:")
print(df.isnull().sum().sum())

=== CLEANED DATASET SUMMARY ===
Total Records : 1500
Total Columns : 24

Unique States: ['Telangana']

District Distribution:
District
Hanamkonda    903
Warangal      523
Hyderabad      59
Karimnagar     15
Name: count, dtype: int64

Missing Values Count:
0


***Export Cleaned Dataset***

In [6]:
# 6. EXPORT CLEANED DATASET & DOWNLOAD

output_filename = "Telangana_NLP_Stress_Sentiment_Dataset_Cleaned.xlsx"

# Export to Excel with openpyxl engine
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Cleaned_Dataset')

print(f"[SUCCESS] Cleaned dataset saved to: {output_filename}")
print("Downloading to local machine...")

# Trigger automatic download in Google Colab
files.download(output_filename)

[SUCCESS] Cleaned dataset saved to: Telangana_NLP_Stress_Sentiment_Dataset_Cleaned.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>